# 🤖 Digital Twin Lab

A **RAG-powered Digital Twin chatbot** for David Inyang-Etoh.

| Phase | What happens |
|-------|--------------|
| **1 — Build Knowledge Base** | Crawl `dinyangetoh.com` + article URLs + extract LinkedIn PDF → save as `.md` files |
| **2 — Vector Ingestion** | Ingest all `.md` files into ChromaDB |
| **3 — RAG Chat** | Retrieve relevant chunks per query → inject as context → generate in-character responses |
| **4 — Gradio + Deploy** | Launch Gradio chat UI locally → export `app.py` for HuggingFace Spaces |

---

**Directory layout:**
```
ai-playground/
├── rag-ai-lab/
│   └── modules/          ← shared modules (crawler, ingester, etc.)
└── digital-twin/         ← this project (self-contained)
    ├── digital-twin-lab.ipynb
    ├── app.py            (generated — for HuggingFace Spaces)
    ├── requirements.txt
    ├── me/
    │   ├── david-inyang-etoh-linkedin.pdf
    │   └── knowledge/    (generated by this notebook)
    │       ├── website/
    │       ├── articles/
    │       └── linkedin.md
    └── db/
        └── digital_twin_db/  (generated ChromaDB)
```
> `modules/` from `rag-ai-lab` are **imported via `sys.path`** — no duplication.

In [5]:
# Cell 2: Install / audit dependencies
!uv pip install playwright beautifulsoup4 pypdf gradio openai python-dotenv chromadb langchain langchain-openai langchain-text-splitters langchain-chroma langchain-community pyyaml pydantic nest-asyncio lxml
!playwright install chromium

Using Python 3.12.12 environment at: /Users/davidinyang-etoh/Projects/ai-projects/ai-playground/.venv
Audited 16 packages in 60ms


In [1]:
# Cell 3: Add shared modules to path (reuse rag-ai-lab/modules without copying)
import sys
import os
import asyncio
from pathlib import Path

# Resolve the path to rag-ai-lab relative to this notebook's location
NOTEBOOK_DIR = Path(os.getcwd())       # directory Jupyter was launched from
# If launched from digital-twin/, parent is ai-playground/
# Adjust if you launch Jupyter from a different directory
RAG_LAB_PATH = NOTEBOOK_DIR.parent / "rag-ai-lab"
if not RAG_LAB_PATH.exists():
    # Fallback: try relative to parent of cwd
    RAG_LAB_PATH = NOTEBOOK_DIR / "../rag-ai-lab"

RAG_LAB_PATH = RAG_LAB_PATH.resolve()

if str(RAG_LAB_PATH) not in sys.path:
    sys.path.insert(0, str(RAG_LAB_PATH))

print(f"✅ Shared modules path: {RAG_LAB_PATH}")
print(f"   modules/ exists: {(RAG_LAB_PATH / 'modules').exists()}")

✅ Shared modules path: /Users/davidinyang-etoh/Projects/ai-projects/ai-playground/rag-ai-lab
   modules/ exists: True


In [2]:
# Cell 4: Imports (modules loaded from rag-ai-lab via sys.path above)
from dotenv import load_dotenv
from openai import OpenAI
from pypdf import PdfReader
import gradio as gr
import nest_asyncio

from modules.crawler import Crawler, CrawlOptions, save_pages
from modules.ingester import (
    Ingester,
    IngestOptions,
    ChunkingOptions,
    VectorStoreOptions,
    EmbeddingOptions,
)

print("✅ Imports OK")

✅ Imports OK


In [19]:
# Cell 5: Environment and personal configuration
load_dotenv(override=True)
nest_asyncio.apply()

openai_client = OpenAI()

# ─── Personal config ──────────────────────────────────────────────────
NAME             = "David Inyang-Etoh"
PERSONAL_WEBSITE = "https://www.dinyangetoh.com"

MY_ARTICLES = [
    "https://medium.com/@dinyangetoh/how-to-build-simple-restful-api-with-nodejs-expressjs-and-mongodb-99348012925d",
    "https://www.linkedin.com/pulse/token-new-currency-david-inyang-etoh-vfjmf",
    "https://www.linkedin.com/pulse/ai-supercar-you-qualified-drive-david-inyang-etoh-u1jif",
]

# ─── Paths (relative to this notebook's directory) ────────────────────
LINKEDIN_PDF  = Path("me/david-inyang-etoh-linkedin.pdf")
RESUME_PDF = Path("me/David_Inyang-Etoh_19032026_Resume.pdf")
KNOWLEDGE_DIR = Path("me/knowledge")       # output: .md knowledge files
VECTOR_DB_DIR = Path("db/digital_twin_db") # output: ChromaDB
CHAT_MODEL    = "gpt-4.1-mini"
EMBED_MODEL   = "text-embedding-3-small"

# Ensure output directories exist
(KNOWLEDGE_DIR / "website").mkdir(parents=True, exist_ok=True)
(KNOWLEDGE_DIR / "articles").mkdir(parents=True, exist_ok=True)

print(f"✅ Config ready")
print(f"   LinkedIn PDF   → {LINKEDIN_PDF}  (exists={LINKEDIN_PDF.exists()})")
print(f"   Resume PDF   → {RESUME_PDF}  (exists={RESUME_PDF.exists()})")
print(f"   Knowledge base → {KNOWLEDGE_DIR}")
print(f"   Vector DB      → {VECTOR_DB_DIR}")

✅ Config ready
   LinkedIn PDF   → me/david-inyang-etoh-linkedin.pdf  (exists=True)
   Resume PDF   → me/David_Inyang-Etoh_19032026_Resume.pdf  (exists=True)
   Knowledge base → me/knowledge
   Vector DB      → db/digital_twin_db


---
## 🌐 Phase 1 — Build Knowledge Base

Crawl the personal website (all reachable internal pages) and each article URL individually.  
Also extract the LinkedIn PDF into markdown.  
All outputs are saved as structured `.md` files under `me/knowledge/`.

In [10]:
# Cell 7: Crawl personal website → me/knowledge/website/
crawl_options = CrawlOptions(
    max_pages=50,
    max_depth=3,
    min_word_count=80,
    split_spa_sections=True,  # handles SPA / single-page portfolio layouts
)

crawler = Crawler()
loop = asyncio.get_event_loop()

print(f"🌐 Crawling: {PERSONAL_WEBSITE} ...")
website_results = loop.run_until_complete(
    crawler.run(url=PERSONAL_WEBSITE, options=crawl_options)
)

summary = website_results.get("summary", {})
print(f"\n✅ Website crawl complete:")
for k, v in summary.items():
    print(f"   {k}: {v}")

🌐 Crawling: https://www.dinyangetoh.com ...

✅ Website crawl complete:
   crawled_pages: 8
   valid_urls: 8
   invalid_urls: 0
   soft_404s: 0
   skipped_assets: 0
   total_time_s: 14.0
   pages_per_second: 0.57


In [11]:
# Cell 8: Save crawled website pages as .md files
website_pages = website_results["data"]["pages"]
saved_count = save_pages(website_pages, str(KNOWLEDGE_DIR / "website"))

print(f"✅ Saved {saved_count} website pages → {KNOWLEDGE_DIR}/website/")
for f in sorted((KNOWLEDGE_DIR / "website").glob("*.md")):
    print(f"   📄 {f.name}  ({f.stat().st_size:,} bytes)")

✅ Saved 8 website pages → me/knowledge/website/
   📄 about.md  (6,593 bytes)
   📄 blog.md  (1,738 bytes)
   📄 blog_aws-cdk-cost-optimization.md  (1,675 bytes)
   📄 blog_cqrs-graphql-performance.md  (1,593 bytes)
   📄 blog_event-driven-microservices-nestjs.md  (2,303 bytes)
   📄 blog_openai-fintech-integration.md  (1,831 bytes)
   📄 index.md  (7,837 bytes)
   📄 portfolio.md  (3,235 bytes)


In [12]:
# Cell 9: Crawl individual article URLs (one page each, graceful fallback on blocks)
article_options = CrawlOptions(
    max_pages=1,
    max_depth=0,
    min_word_count=50,
    split_spa_sections=False,
)

all_article_pages = []
failed_articles = []

for url in MY_ARTICLES:
    print(f"\n📰 Fetching: {url}")
    try:
        result = loop.run_until_complete(
            crawler.run(url=url, options=article_options)
        )
        pages = result["data"]["pages"]
        all_article_pages.extend(pages)
        print(f"   → {len(pages)} page(s) extracted, {result['summary']['total_time_s']}s")
    except Exception as e:
        print(f"   ⚠️  Failed (likely login-gated): {e}")
        failed_articles.append(url)

print(f"\n✅ Total article pages extracted: {len(all_article_pages)}")
if failed_articles:
    print(f"⚠️  Blocked URLs ({len(failed_articles)}): add content manually to me/knowledge/articles/")
    for u in failed_articles:
        print(f"   - {u}")


📰 Fetching: https://medium.com/@dinyangetoh/how-to-build-simple-restful-api-with-nodejs-expressjs-and-mongodb-99348012925d


[  1/1] ERROR  https://medium.com/@dinyangetoh/how-to-build-simple-restful-api-with-nodejs-expressjs-and-mongodb-99348012925d  Page.goto: Timeout 30000ms exceeded.
Call log:
  - navigating to "https://medium.com/@dinyangetoh/how-to-build-simple-restful-api-with-nodejs-expressjs-and-mongodb-99348012925d", waiting until "networkidle"



   → 0 page(s) extracted, 94.8s

📰 Fetching: https://www.linkedin.com/pulse/token-new-currency-david-inyang-etoh-vfjmf
   → 1 page(s) extracted, 36.3s

📰 Fetching: https://www.linkedin.com/pulse/ai-supercar-you-qualified-drive-david-inyang-etoh-u1jif
   → 1 page(s) extracted, 9.2s

✅ Total article pages extracted: 2


In [13]:
# Cell 10: Save article pages as .md files
if all_article_pages:
    saved_count = save_pages(all_article_pages, str(KNOWLEDGE_DIR / "articles"))
    print(f"✅ Saved {saved_count} article pages → {KNOWLEDGE_DIR}/articles/")
    for f in sorted((KNOWLEDGE_DIR / "articles").glob("*.md")):
        print(f"   📄 {f.name}  ({f.stat().st_size:,} bytes)")
else:
    print("⚠️  No article pages to save.")
    print("   You can manually add .md files to me/knowledge/articles/ and re-run Phase 2+.")

✅ Saved 2 article pages → me/knowledge/articles/
   📄 pulse_ai-supercar-you-qualified-drive-david-inyang-etoh-u1jif.md  (1,042 bytes)
   📄 pulse_token-new-currency-david-inyang-etoh-vfjmf.md  (953 bytes)


In [4]:
# Cell 11: Extract LinkedIn PDF → me/knowledge/linkedin.md
def pdf_to_markdown(pdf_path: Path, person_name: str,type: str = "linkedin" or "resume") -> str:
    """Extract all PDF pages and wrap in structured markdown with YAML frontmatter."""
    reader = PdfReader(str(pdf_path))
    text_parts = [
        page.extract_text().strip()
        for page in reader.pages
        if page.extract_text() and page.extract_text().strip()
    ]
    full_text = "\n\n".join(text_parts)

    if type == "linkedin":
        return (
            f"---\n"
            f"title: LinkedIn Profile — {person_name}\n"
            f"source: linkedin_pdf\n"
            f"breadcrumb: LinkedIn Profile\n"
            f"---\n\n"
            f"# {person_name} — LinkedIn Profile\n\n"
            f"{full_text}\n"
        )
    elif type == "resume":
        return (
            f"---\n"
            f"title: Resume — {person_name}\n"
            f"source: resume_pdf\n"
            f"breadcrumb: Resume\n"
            f"---\n\n"
            f"# {person_name} — Resume\n\n"
            f"{full_text}\n"
        )

    else:
        raise ValueError(f"Invalid type: {type}")

if LINKEDIN_PDF.exists():
    linkedin_md = pdf_to_markdown(LINKEDIN_PDF, NAME, type="linkedin")
    out_path = KNOWLEDGE_DIR / "linkedin.md"
    out_path.write_text(linkedin_md, encoding="utf-8")
    print(f"✅ LinkedIn PDF extracted → {out_path}")
    print(f"   {len(linkedin_md):,} characters, {len(linkedin_md.splitlines())} lines")
else:
    print(f"⚠️  PDF not found: {LINKEDIN_PDF}")

if RESUME_PDF.exists():
    resume_md = pdf_to_markdown(RESUME_PDF, NAME, type="resume")
    out_path = KNOWLEDGE_DIR / "resume.md"
    out_path.write_text(resume_md, encoding="utf-8")
    print(f"✅ Resume PDF extracted → {out_path}")
    print(f"   {len(resume_md):,} characters, {len(resume_md.splitlines())} lines")
else:
    print(f"⚠️  PDF not found: {RESUME_PDF}")

✅ LinkedIn PDF extracted → me/knowledge/linkedin.md
   6,012 characters, 139 lines
✅ Resume PDF extracted → me/knowledge/resume.md
   6,350 characters, 508 lines


In [5]:
# Cell 12: Inspect the full knowledge base
md_files = sorted(KNOWLEDGE_DIR.rglob("*.md"))
total_bytes = sum(f.stat().st_size for f in md_files)

print(f"📚 Knowledge base: {len(md_files)} files | {total_bytes:,} bytes total\n")
print(f"{'File':<50} {'Size':>12}")
print("-" * 64)
for f in md_files:
    rel = str(f.relative_to(KNOWLEDGE_DIR))
    print(f"{rel:<50} {f.stat().st_size:>10,} bytes")

📚 Knowledge base: 12 files | 41,328 bytes total

File                                                       Size
----------------------------------------------------------------
articles/pulse_ai-supercar-you-qualified-drive-david-inyang-etoh-u1jif.md      1,042 bytes
articles/pulse_token-new-currency-david-inyang-etoh-vfjmf.md        953 bytes
linkedin.md                                             6,120 bytes
resume.md                                               6,408 bytes
website/about.md                                        6,593 bytes
website/blog.md                                         1,738 bytes
website/blog_aws-cdk-cost-optimization.md               1,675 bytes
website/blog_cqrs-graphql-performance.md                1,593 bytes
website/blog_event-driven-microservices-nestjs.md       2,303 bytes
website/blog_openai-fintech-integration.md              1,831 bytes
website/index.md                                        7,837 bytes
website/portfolio.md                     

---
## 🗄️ Phase 2 — Vector Ingestion

Ingest all `.md` files from `me/knowledge/` into a **ChromaDB vector store** at `db/digital_twin_db/`.

In [6]:
# Cell 14: Configure and run the Ingester
ingest_options = IngestOptions(
    knowledge_base_path=str(KNOWLEDGE_DIR),
    chunking=ChunkingOptions(
        chunk_size=1000,
        chunk_overlap=150,
        min_chunk_chars=60,
    ),
    vector_store=VectorStoreOptions(
        persist_directory=str(VECTOR_DB_DIR),
        delete_existing_collection=True,  # fresh rebuild each run
    ),
    embedding=EmbeddingOptions(model=EMBED_MODEL),
)

ingester = Ingester(ingest_options)

print(f"⏳ Ingesting {KNOWLEDGE_DIR} ...")
vectorstore, result = ingester.ingest()

print(f"\n✅ Ingestion complete:")
print(f"   Vectors:    {result.vector_count:,}")
print(f"   Dimensions: {result.embedding_dimensions:,}")
print(f"   Persisted → {VECTOR_DB_DIR}")

delete_existing_collection=True — wiping vector store at db/digital_twin_db


⏳ Ingesting me/knowledge ...

✅ Ingestion complete:
   Vectors:    53
   Dimensions: 1,536
   Persisted → db/digital_twin_db


In [7]:
# Cell 15: Verify — test similarity search with sample queries
from IPython.display import Markdown, display

test_queries = [
    "What does David do professionally?",
    "What projects has David worked on?",
    "What are David's technical skills?",
]

for query in test_queries:
    hits = ingester.similarity_search(query, k=2)
    print(f"\n🔍 '{query}'")
    for i, h in enumerate(hits, 1):
        source = h.metadata.get("source", "?")
        title  = h.metadata.get("page_title", h.metadata.get("title", ""))
        preview = h.page_content[:120].replace("\n", " ")
        print(f"   [{i}] {Path(source).name} | {title}")
        print(f"       {preview}...")


🔍 'What does David do professionally?'
   [1] index.md | David Inyang-Etoh — Lead Software Engineer & AI Systems Builder
       [www.dinyangetoh.com]  working with David for 2 years at receeve, where he was in one of my teams as a Senior Full Stack...
   [2] linkedin.md | LinkedIn Profile — David Inyang-Etoh
       [LinkedIn Profile]  # David Inyang-Etoh — LinkedIn Profile  Contact dinyangetoh@gmail.com www.linkedin.com/in/david-inya...

🔍 'What projects has David worked on?'
   [1] index.md | David Inyang-Etoh — Lead Software Engineer & AI Systems Builder
       [www.dinyangetoh.com]  working with David for 2 years at receeve, where he was in one of my teams as a Senior Full Stack...
   [2] index.md | David Inyang-Etoh — Lead Software Engineer & AI Systems Builder
       [www.dinyangetoh.com]  Lead Software Engineer · Node.js · TypeScript · AWS · AI/ML I design and build scalable backend s...

🔍 'What are David's technical skills?'
   [1] index.md | David Inyang-Etoh — Lead Software 

---
## 💬 Phase 3 — RAG Chat

Retrieves the top-k most relevant chunks from ChromaDB for each user message and injects them as context.

In [8]:
# Cell 17: Base system prompt
BASE_SYSTEM_PROMPT = f"""You are acting as {NAME} — my digital twin.
You answer questions about {NAME}'s career, skills, projects, writing, and professional interests.
Always speak in first person ("I", "my", "I've") and stay in character throughout.
Be warm, genuine, and professionally engaging — as if {NAME} is personally chatting.
Draw on the context provided to give specific, accurate answers.
If something is genuinely not in your context, say so honestly rather than guessing."""

print("✅ System prompt set")

✅ System prompt set


In [13]:
# Cell 18: RAG-powered chat function
def build_rag_system_prompt(query: str, k: int = 5) -> str:
    """Retrieve top-k chunks for the query and inject into the system prompt."""
    chunks = ingester.similarity_search(query, k=k)
    if not chunks:
        return BASE_SYSTEM_PROMPT
    context_blocks = [
        f"[Source: {Path(c.metadata.get('source', 'unknown')).name}]\n{c.page_content}"
        for c in chunks
    ]
    context = "\n\n---\n\n".join(context_blocks)
    return BASE_SYSTEM_PROMPT + f"\n\n## Relevant Context\n\n{context}"


def chat(message: str, history: list) -> str:
    """RAG chat: retrieve relevant context per message, then generate a response."""
    system_prompt = build_rag_system_prompt(message)
    clean_history = [
        {"role": h["role"], "content": h["content"]} for h in history
    ]
    messages = (
        [{"role": "system", "content": system_prompt}]
        + clean_history
        + [{"role": "user", "content": message}]
    )
    response = openai_client.chat.completions.create(
        model=CHAT_MODEL, messages=messages
    )
    return response.choices[0].message.content


# Quick sanity check
test_reply = chat("What's your professional background?", [])
print("✅ Chat function working. Sample response:")
print("-" * 60)
print(test_reply[:500])

✅ Chat function working. Sample response:
------------------------------------------------------------
I have over 8 years of experience as a Lead Software Engineer, focusing primarily on building and scaling SaaS and FinTech platforms across Europe, the US, and Africa. Currently, I lead the backend and cloud infrastructure at Collectwire Technologies, where I've successfully architected and delivered a global payroll SaaS MVP within a tight timeline.

Throughout my career, I've specialized in cloud-native, event-driven microservices. I guide teams in moving from monolithic systems to more distri


---
## 🎨 Phase 4 — Gradio UI & HuggingFace Spaces Deployment

In [17]:
# Cell 20: Short page blurb + in-chat welcome
CHAT_DESCRIPTION = (
    f"Chat with Mr Dee — a RAG-powered digital twin of {NAME}. "
    "Ask questions grounded in his knowledge base."
)
WELCOME_MESSAGE = f"""👋 Hi! I'm Mr Dee, the digital twin of **{NAME}** — software engineer, AI builder, and technical writer.

I can answer questions about my background, projects, professional experience, skills, and the things I write and speak about.

Feel free to ask me anything!"""

In [18]:
# Cell 21: Launch Gradio Chat UI (local preview)
demo = gr.ChatInterface(
    fn=chat,
    title=f"🤖 {NAME} — Digital Twin",
    description=CHAT_DESCRIPTION,
    chatbot=gr.Chatbot(
        value=[{"role": "assistant", "content": WELCOME_MESSAGE}],
    ),
    examples=[
        "Tell me about your background and experience",
        "What projects are you currently working on?",
        "What's your take on AI and its impact on software?",
        "What technologies do you specialize in?",
        "How can I get in touch with you?",
    ],
)

demo.launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


In [20]:
# Cell 22: Export standalone app.py for HuggingFace Spaces
# On HF Spaces: modules/ is uploaded alongside app.py — no sys.path trick needed.
APP_PY_TEMPLATE = '''
import os
from pathlib import Path

from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

from modules.ingester import (
    Ingester, IngestOptions, ChunkingOptions, VectorStoreOptions, EmbeddingOptions,
)

load_dotenv(override=True)
client = OpenAI()

NAME          = "{name}"
KNOWLEDGE_DIR = Path("me/knowledge")
VECTOR_DB_DIR = Path("db/digital_twin_db")
CHAT_MODEL    = "gpt-4.1-mini"
EMBED_MODEL   = "text-embedding-3-small"

# Load pre-built vector store (no re-embedding needed on HF Spaces)
ingester = Ingester(IngestOptions(
    knowledge_base_path=str(KNOWLEDGE_DIR),
    chunking=ChunkingOptions(chunk_size=1000, chunk_overlap=150, min_chunk_chars=60),
    vector_store=VectorStoreOptions(persist_directory=str(VECTOR_DB_DIR), delete_existing_collection=False),
    embedding=EmbeddingOptions(model=EMBED_MODEL),
))
_, _ = ingester.ingest()

BASE_PROMPT = (
    f"You are acting as {{NAME}} — their digital twin. "
    f"Answer questions about {{NAME}}'s career, skills, projects, and writing. "
    "Speak in first person and stay in character. Be warm and engaging. "
    "If something is not in your context, say so honestly."
)

def chat(message, history):
    chunks = ingester.similarity_search(message, k=5)
    context = "\n\n---\n\n".join(
        f"[{{Path(c.metadata.get('source', '')).name}}]\n{{c.page_content}}" for c in chunks
    )
    system = BASE_PROMPT + (f"\n\n## Context\n\n{{context}}" if context else "")
    msgs = (
        [{{"role": "system", "content": system}}]
        + [{{"role": h["role"], "content": h["content"]}} for h in history]
        + [{{"role": "user", "content": message}}]
    )
    return client.chat.completions.create(model=CHAT_MODEL, messages=msgs).choices[0].message.content

CHAT_DESCRIPTION = (
    f"Chat with Mr Dee — a RAG-powered digital twin of {{NAME}}. "
    "Ask questions grounded in his knowledge base."
)
WELCOME_MESSAGE = f"""👋 Hi! I'm Mr Dee, the digital twin of **{{NAME}}** — software engineer, AI builder, and technical writer.

I can answer questions about my background, projects, professional experience, skills, and the things I write and speak about.

Feel free to ask me anything!"""

demo = gr.ChatInterface(
    fn=chat,
    title=f"🤖 {{NAME}} — Digital Twin",
    description=CHAT_DESCRIPTION,
    chatbot=gr.Chatbot(
        value=[{{"role": "assistant", "content": WELCOME_MESSAGE}}],
    ),
    examples=[
        "Tell me about your background",
        "What are you building?",
        "What's your view on AI?",
        "What technologies do you specialize in?",
    ],
    theme=gr.themes.Soft(),
)

if __name__ == "__main__":
    demo.launch()

'''

app_py = APP_PY_TEMPLATE.format(name=NAME).strip()
Path("app.py").write_text(app_py, encoding="utf-8")
print("✅ app.py written — ready for HuggingFace Spaces")
print(f"   {len(app_py):,} characters")


✅ app.py written — ready for HuggingFace Spaces
   2,645 characters


---
## 🚀 HuggingFace Spaces Deployment

### Files to push to your HF Space:

| File / Folder | Notes |
|---------------|-------|
| `app.py` | Generated by Cell 22 |
| `modules/` | Copy from `../rag-ai-lab/modules/` |
| `me/knowledge/` | All `.md` knowledge files |
| `db/digital_twin_db/` | Pre-built ChromaDB (no re-embedding on startup) |
| `requirements.txt` | Python dependencies |

### Steps:

```bash
# 1. Login
huggingface-cli login

# 2. Create Space (Gradio SDK)
huggingface-cli repo create digital-twin --type space --space_sdk gradio

# 3. Clone and push
git clone https://huggingface.co/spaces/<your-username>/digital-twin
cd digital-twin
# copy app.py, modules/, me/, db/, requirements.txt here
git add . && git commit -m 'Deploy digital twin' && git push
```

**Add Secret** → `OPENAI_API_KEY` in Space Settings > Variables and Secrets

> **Tip**: Because `db/digital_twin_db/` is pre-built and uploaded, HF Spaces loads instantly — no embedding costs on startup.